# R1 — Self-Supervised Morphologyy Representation Learning Pipeline.

**Domain:** Radio Astronomy
**Status:** Planning (Phase 0)  
**Deliverables planned:** 2 Zenodo deposits, 1 RNAAS note

---

## Abstract

Public radio continuum surveys — VLA/FIRST, LOFAR/LoTSS, ASKAP/EMU-RACS, MeerKAT/MIGHTEE — are each well curated individually, but no reproducible, ML-ready product currently harmonizes them into a single dataset with consistent cutouts, flux units, resolution handling, and metadata conventions. As a result, every group building machine-learning models on radio imaging data re-implements its own bespoke preprocessing, which makes cross-survey ML results difficult to compare or reproduce.

This project builds a reproducible pipeline that ingests, quality-controls, harmonizes, and benchmarks cutouts and catalogs drawn from 2–4 public radio surveys, releasing the resulting dataset and pipeline as open, citable research products. The work is scoped as a data-engineering thesis: the contribution is not a new scientific result about radio sources, but a validated, documented, reusable *resource* that lowers the barrier to future radio-ML research (including projects R2 and R3 in this portfolio).

## 1. Introduction and Research Gap

Modern radio continuum surveys differ substantially in angular resolution, frequency coverage, sensitivity, pixel scale, calibration convention, and catalog schema. Studies that combine or compare sources across surveys (e.g., for morphology classification, cross-identification, or representation learning) must therefore solve the same low-level data-wrangling problem — reprojecting cutouts, reconciling flux units, matching noise estimation conventions — largely from scratch each time.

This duplicated effort has two consequences relevant to reproducible ML research:

1. **Non-comparability.** Two papers reporting different classification accuracies on "the same" survey may in fact be evaluating on differently preprocessed inputs.
2. **High barrier to entry.** New ML methods cannot easily be tested across surveys without first re-solving the harmonization problem.

**Research gap.** There is no publicly released, versioned, documented pipeline (nor its output dataset) that performs this harmonization end-to-end and exposes the result as a ready-to-use ML benchmark with traceable provenance back to survey-native catalogs.

This project addresses that gap directly and is deliberately scoped as *infrastructure*: its success criterion is reuse-readiness, not a novel astrophysical finding.

## 2. Research Question

> **How can heterogeneous radio survey products be transformed into a unified, reproducible dataset suitable for downstream ML research?**

Sub-questions to be answered by the pipeline's design and validation:

- What minimal set of metadata fields must be preserved/normalized to keep cross-survey comparisons valid (beam shape, frequency, epoch, noise level)?
- What quality-control criteria reliably separate usable cutouts from artefact- or RFI-dominated ones, and how well do simple statistical/CNN-based classifiers agree with visual inspection?
- How closely do harmonized, pipeline-derived flux/position measurements reproduce the values reported in each survey's native catalog (i.e., what is the harmonization error budget)?

## 3. Objectives

- **O1 — Acquisition:** Assemble a documented, provenance-tracked corpus of cutouts and catalogs from 2–4 public radio surveys.
- **O2 — Quality control:** Establish automated flagging of artefacts, sidelobes, and residual RFI, with a per-cutout noise/rms estimate.
- **O3 — Harmonization:** Normalize units, beam sizes, and pixel scales; extract a common tabular feature set (peak flux, integrated flux, size, ellipticity) into one FITS/HDF5 schema.
- **O4 — Benchmarking:** Produce documented train/validation/test splits, validated against survey-native catalogs, packaged as a citable, reusable dataset.

## 4. Candidate Data Sources

| Survey | Instrument | Access point | Notes (resolution / frequency / area) |
|---|---|---|---|
| FIRST | VLA | CIRADA / CADC cutout service | *TBD — fill in during Phase 1* |
| LoTSS DR2 | LOFAR | LOFAR Surveys archive | *TBD* |
| EMU / RACS | ASKAP | CASDA / CIRADA | *TBD* |
| MIGHTEE | MeerKAT | SARAO archive | *TBD* |

Final survey selection (2–4 of the above) and the criteria used to choose them should be recorded here once Phase 1 begins — this table is the seed of the dataset's provenance record.

## 5. Methodology — Four-Phase Work Plan

The project is executed in four sequential phases. Each phase below states its objective, concrete tasks, methods/tools, and the artefact it must produce before the next phase begins. Corresponding code should live in sections mirroring this structure (see §9, Notebook Roadmap).

### Phase 1 — Survey Selection and Acquisition

**Objective.** Establish a reproducible, provenance-logged corpus of raw survey products.

**Tasks**
- Select 2–4 public surveys with open image/catalog access.
- Script bulk retrieval via cutout services or the CIRADA/CADC/LOFAR-Surveys archives.
- Log provenance for every file: survey, data release, beam, epoch, field/tile identifier.

**Methods / tools.** Archive APIs (CIRADA, CADC), `astroquery`, bulk FITS retrieval scripts.

**Phase output.** A raw-data manifest (table) with one row per file and complete provenance columns; this manifest is the traceability backbone for every later validation step.

### Phase 2 — Quality Control and RFI Mitigation

**Objective.** Separate usable cutouts from artefact-, sidelobe-, or RFI-dominated ones.

**Tasks**
- Apply automated flagging for artefacts, sidelobes, and residual RFI.
- Establish a per-cutout noise/rms estimation procedure.
- Discard or flag low-quality tiles using statistical thresholds or a lightweight CNN-based quality classifier.

**Methods / tools.** Local rms/sigma-clipping statistics; optional small CNN classifier trained on a hand-labeled subset for QC.

**Phase output.** A QC-annotated manifest (quality flag + rms per cutout) and a short report quantifying the fraction flagged, with example accept/reject cutouts for visual sanity-checking.

### Phase 3 — Metadata Harmonization and Feature Extraction

**Objective.** Bring all surviving cutouts and catalog entries into one common, unit-consistent schema.

**Tasks**
- Normalize flux units, beam sizes, and pixel scales across surveys.
- Extract standard tabular features per source: peak flux, integrated flux, angular size, ellipticity.
- Define and implement a common FITS/HDF5 schema for cutouts + tabular features.

**Methods / tools.** `astropy` (units, WCS reprojection), `reproject`, pandas/HDF5 schema design.

**Phase output.** The harmonized dataset in its final schema, plus a schema/data-dictionary document.

### Phase 4 — Benchmark Construction and Validation

**Objective.** Turn the harmonized dataset into a validated, documented, reusable ML benchmark.

**Tasks**
- Assemble train/validation/test splits with documented, non-leaking selection criteria.
- Validate harmonized flux/position measurements against each survey's native catalog (quantify agreement/error budget — this directly answers the research sub-question in §2).
- Write full data-card documentation (scope, limitations, intended use, known biases).

**Methods / tools.** Cross-matching against native catalogs (positional + statistical agreement checks), data-card template (cf. Hugging Face / ML-research data-card conventions).

**Phase output.** The release-ready dataset, split definitions, validation report, and data card — these become the content of the Zenodo deposits below.

## 6. Deliverables

**Zenodo Deposit 1 — Radio Foundation Dataset.** Harmonized cutouts + catalog, versioned, with DOI.

**Zenodo Deposit 2 — Processing Pipeline + Documentation.** Code, configuration files, and the data card, versioned separately from the dataset so the pipeline can be re-run against future survey data releases.

**RNAAS Note — "Release of a reproducible ML-ready radio survey resource."** A short (≤1000-word) note announcing the dataset and pipeline, summarizing scope, validation results (Phase 4), and intended reuse; cites both Zenodo DOIs.

## 7. Validation and Reproducibility Plan

- **External validation:** cross-check harmonized flux/position values against survey-native catalogs (Phase 4); report agreement statistics, not just "done/not done".
- **Internal reproducibility:** pipeline must be re-runnable end-to-end from the raw-data manifest (Phase 1) given only public archive access — no manual/undocumented steps.
- **Documentation standard:** data card covering provenance, known limitations, and intended use, released alongside the dataset (not as an afterthought).
- **Versioning:** dataset and pipeline are versioned independently on Zenodo so future survey data releases (e.g., a new LoTSS DR) can be re-harmonized without invalidating the original citable snapshot.

## 8. Related Work and Starting Points

- N. Gupta et al., *"Machine Learning Frameworks for Large-Scale Radio Surveys"* (arXiv:2510.11145) — overview of the EMU pipeline (RadioGalaxyNET, Gal-DINO, RG-CAT, EMUSE); template for an end-to-end survey-to-ML-product pipeline.
- CSIRO/ICRAR EMU team (R. Norris, N. Gupta, O. I. Wong) — reproducible ASKAP/EMU catalogue pipelines (RG-CAT, Gal-DINO) as a technical starting point.
- LOFAR Surveys collaboration (W. Williams, M. Hardcastle, T. Shimwell) — LoTSS DR1/DR2 papers as a documented preprocessing/cataloguing convention to adopt.
- CIRADA documentation — cutout services, cross-matching infrastructure, and metadata standards across ASKAP/VLA surveys.

*A systematic ADS/arXiv literature search should be carried out once the specific survey subset is fixed in Phase 1 — the above is a starting map, not an exhaustive review.*

## 9. Notebook Roadmap (for implementation)

This notebook currently contains planning content only. As implementation proceeds, code sections should be added in this order, each ending with a short markdown checkpoint summarizing what was produced and how it was validated:

1. **Setup** — imports, config, path definitions.
2. **Phase 1 code** — acquisition scripts + raw-data manifest.
3. **Phase 2 code** — QC/RFI flagging + rms estimation.
4. **Phase 3 code** — harmonization + feature extraction into the common schema.
5. **Phase 4 code** — split construction, native-catalog validation, data-card generation.
6. **Release checklist** — final checks before Zenodo upload and RNAAS drafting.

## 10. Future Relevance

This dataset and pipeline are the shared foundation for **R2** (self-supervised radio representation learning) and **R3** (radio source knowledge graph), both of which consume R1's harmonized cutouts/catalog as their Phase-1 input. Beyond this portfolio, the pipeline's design questions (survey harmonization, provenance, reproducible QC) map directly onto SKAO/NRAO/ASTRON-scale data-infrastructure work — relevant future-PhD context if pursuing that direction.